In [1]:
import numpy as np
from pathlib import Path
import pandas as pd
import statsmodels.formula.api as smf

from src.processing import read_txt_file, read_excel, data_summary, id_summary, get_sample_summary, merge_data
from src.descriptive import create_sample_overview, create_missing_values_table, create_numeric_summary, create_categorical_summary
from src.plotting import plot_distribution, plot_relationship
from src.results import extract_regression_results, create_results_table

In [2]:
pd.set_option("display.max_columns", None)

Path("results").mkdir(parents=True, exist_ok=True)

## A. Datenverarbeitung
### Sample-Daten

In diesen Daten ist ein Mapping von Patienten-ID zu Subject-ID sowie RNA- und DNA-Sample-IDs hinterlegt. Außerdem gibt die Spalte rna_control Auskunft darüber, ob es sich um gesunde Kontrollproben handelt.

In [3]:
sample_data = read_excel("beataml_waves1to4_sample_mapping.xlsx")

sample_data.shape

(979, 12)

In [4]:
data_summary(sample_data)

,patientId,dbgap_subject_id,labId,dbgap_rnaseq_sample,rna_seq_id,rna_control,rna_include_in_analysis,dbgap_dnaseq_sample,dna_seq_id,dna_capture_type,dna_include_in_analysis,analysisDrug
dtype,int64,int64,str,str,str,str,str,str,str,str,str,str
n missing,0,0,0,244,244,244,244,76,76,76,76,37
p% missing,0.0,0.0,0.0,24.92,24.92,24.92,24.92,7.76,7.76,7.76,7.76,3.78
n unique,826,826,979,735,735,4,2,903,903,2,2,2


In den Sample-Daten sind gesunde Kontrollproben, Proben mit dem Wert „No“ und Proben mit fehlenden Werten in rna_control enthalten. Ich schließe die eindeutig gekennzeichneten gesunden Kontrollproben an dieser Stelle aus. Ob es sich bei den übrigen Proben um AML-Proben handelt, prüfe ich später anhand der klinischen Daten.

In [5]:
id_summary(sample_data)

,Anzahl
n Patienten,826
n RNA Seq,736


In [6]:
sample_data["rna_control"].value_counts(dropna=False)

rna_control
No                           698
NaN                          244
Healthy Individual BM MNC     20
Healthy pooled CD34+          16
Healthy Individual CD34+       1
Name: count, dtype: int64

In [7]:
# Filtern der Proben auf Kontrolle: Nein oder NaN
sample_data = sample_data[sample_data["rna_control"].isin(["No", np.nan])]

In [8]:
sample_data["rna_control"].value_counts(dropna=False)

rna_control
No     698
NaN    244
Name: count, dtype: int64

In [9]:
id_summary(sample_data)

,Anzahl
n Patienten,805
n RNA Seq,699


### Response-Daten

Die Response-Daten enthalten auf Probenebene die AUC der Dosis-Wirkungs-Kurven für verschiedene Inhibitoren. Für meine Analyse filtere ich die Tabelle auf Venetoclax.

Die Tabelle enthält außerdem weitere Modellparameter. Für die weitere Analyse ist vor allem relevant, ob das jeweilige Dosis-Wirkungs-Modell konvergiert ist.

In [10]:
response_data = read_txt_file("beataml_probit_curve_fits_v4_dbgap.txt")

# Umbenennen der auc Spalte zu AUC da es als Label für spätere Plots verwendet wird
response_data = response_data.rename(columns={"auc": "AUC"})

response_data.shape

(63395, 26)

In [11]:
data_summary(response_data)

,dbgap_subject_id,dbgap_dnaseq_sample,dbgap_rnaseq_sample,inhibitor,type,status,paper_inclusion,min_conc,max_conc,intercept,beta,beta_z,beta_p,aic,pearson_chisq,deviance,converged,ic10,ic25,ic50,ic75,ic90,AUC,all_gt_50,all_lt_50,curve_type
dtype,int64,str,str,str,str,str,bool,float64,int64,float64,float64,float64,float64,float64,float64,float64,bool,float64,float64,float64,float64,float64,float64,bool,bool,str
n missing,0,2652,7714,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
p% missing,0.0,4.18,12.17,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
n unique,569,603,542,166,1,3,1,8,5,61711,61711,61710,61703,61698,61705,61699,2,26416,32707,27007,16641,9138,61706,2,2,2


In [12]:
id_summary(response_data)

,Anzahl
n Patienten,569
n RNA Seq,543


In [13]:
response_data[response_data["inhibitor"].isin(["Venetoclax", "Metformin", "Etomoxir"])]["inhibitor"].value_counts()

inhibitor
Venetoclax    430
Etomoxir       52
Metformin      50
Name: count, dtype: int64

Für Metformin und Etomoxir liegen deutlich weniger Proben mit verfügbaren Response-Daten vor als für Venetoclax. Da die Fallzahl für eine separate Analyse zu gering ist, beschränke ich die weitere Untersuchung auf Venetoclax.

In [14]:
# Filtern auf Venetoclax
response_data = response_data[response_data["inhibitor"].eq("Venetoclax")]

In [15]:
id_summary(response_data)

,Anzahl
n Patienten,390
n RNA Seq,383


### Klinische Daten

Die klinischen Daten enthalten detaillierte Patienten- und Krankheitsdaten, sowohl zur Diagnosestellung als auch zum Zeitpunkt der Probenentnahme und zum letzten bekannten Zeitpunkt.

cohort beschreibt die Zugehörigkeit zu einer Kohorte und kann aufgrund möglicher systematischer Unterschiede zwischen den Kohorten für die Adjustierung des Modells relevant sein.
- Waves 1+2: die ersten beiden Datenerhebungswellen, die bereits in der ursprünglichen Beat-AML-Studie beschrieben wurden.
- Waves 3+4: die späteren Datenerhebungswellen mit zusätzlichen Proben, darunter neue Patienten und weitere Proben bereits erfasster Patienten.

diseaseStageAtSpecimenCollection beschreibt das Krankheitsstadium zum Zeitpunkt der Probenentnahme und kann ebenfalls für die Adjustierung des Modells relevant sein.

dxAtSpecimenAcquisition und specificDxAtAcquisition erfassen die Diagnose zum Zeitpunkt der Probenentnahme auf unterschiedlichem Detaillierungsniveau. Diese Variablen nutze ich, um die für meine Fragestellung relevanten AML-Proben auszuwählen.

In [16]:
clinical_data = read_excel("beataml_wv1to4_clinical.xlsx")

In [17]:
data_summary(clinical_data)

,dbgap_subject_id,dbgap_dnaseq_sample,dbgap_rnaseq_sample,cohort,used_manuscript_analyses,manuscript_dnaseq,manuscript_rnaseq,manuscript_inhibitor,consensus_sex,inferred_sex,reportedRace,reportedEthnicity,inferred_ethnicity,centerID,CEBPA_Biallelic,consensusAMLFusions,ageAtDiagnosis,isRelapse,isDenovo,isTransformed,specificDxAtAcquisition_MDSMPN,nonAML_MDSMPN_specificDxAtAcquisition,priorMalignancyNonMyeloid,priorMalignancyType,cumulativeChemo,priorMalignancyRadiationTx,priorMDS,priorMDSMoreThanTwoMths,priorMDSMPN,priorMDSMPNMoreThanTwoMths,priorMPN,priorMPNMoreThanTwoMths,dxAtInclusion,specificDxAtInclusion,ELN2017,dxAtSpecimenAcquisition,specificDxAtAcquisition,ageAtSpecimenAcquisition,timeOfSampleCollectionRelativeToInclusion,specimenGroups,diseaseStageAtSpecimenCollection,specimenType,rnaSeq,exomeSeq,totalDrug,analysisRnaSeq,analysisExomeSeq,analysisDrug,cumulativeTreatmentTypeCount,cumulativeTreatmentTypes,cumulativeTreatmentRegimenCount,cumulativeTreatmentRegimens,cumulativeTreatmentStageCount,cumulativeTreatmentStages,responseToInductionTx,typeInductionTx,responseDurationToInductionTx,mostRecentTreatmentType,currentRegimen,currentStage,mostRecentTreatmentDuration,vitalStatus,overallSurvival,causeOfDeath,%.Basophils.in.PB,%.Blasts.in.BM,%.Blasts.in.PB,%.Eosinophils.in.PB,%.Immature.Granulocytes.in.PB,%.Lymphocytes.in.PB,%.Monocytes.in.PB,%.Neutrophils.in.PB,%.Nucleated.RBCs.in.PB,ALT,AST,albumin,creatinine,fabBlastMorphology,hematocrit,hemoglobin,karyotype,LDH,MCV,otherCytogenetics,plateletCount,surfaceAntigensImmunohistochemicalStains,totalProtein,wbcCount,FLT3-ITD,allelic_ratio,NPM1,RUNX1,ASXL1,TP53,variantSummary
dtype,int64,str,str,str,str,str,str,str,str,str,str,str,str,int64,str,str,float64,str,str,str,bool,bool,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,float64,int64,str,str,str,str,str,str,str,str,str,float64,str,float64,str,float64,str,str,str,float64,str,str,str,float64,str,int64,str,float64,str,str,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,str,float64,float64,str,float64,float64,str,float64,str,float64,float64,str,float64,str,str,str,str,str
n missing,0,39,244,0,0,0,0,0,1,61,283,409,272,0,896,774,26,0,0,0,0,0,0,767,0,0,0,0,0,0,0,0,0,0,0,0,0,16,0,8,29,0,0,0,0,0,0,0,2,78,2,78,2,78,195,192,192,2,2,2,79,0,0,32,430,200,373,397,705,367,317,370,599,391,393,393,419,483,356,342,100,567,437,236,274,248,408,168,3,78,4,821,842,855,319
p% missing,0.0,4.14,25.9,0.0,0.0,0.0,0.0,0.0,0.11,6.48,30.04,43.42,28.87,0.0,95.12,82.17,2.76,0.0,0.0,0.0,0.0,0.0,0.0,81.42,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.7,0.0,0.85,3.08,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.21,8.28,0.21,8.28,0.21,8.28,20.7,20.38,20.38,0.21,0.21,0.21,8.39,0.0,0.0,3.4,45.65,21.23,39.6,42.14,74.84,38.96,33.65,39.28,63.59,41.51,41.72,41.72,44.48,51.27,37.79,36.31,10.62,60.19,46.39,25.05,29.09,26.33,43.31,17.83,0.32,8.28,0.42,87.15,89.38,90.76,33.86
n unique,805,903,698,3,2,2,2,2,2,3,8,4,8,8,2,8,83,3,3,3,2,2,2,51,2,2,2,2,2,3,2,2,6,38,9,6,36,82,135,33,5,3,2,2,2,2,2,2,6,42,11,497,9,212,6,4,84,10,145,20,145,3,579,5,38,137,131,57,54,309,221,262,52,109,99,34,139,16,212,90,481,311,231,398,195,669,45,623,2,70,2,120,98,86,550


In [18]:
clinical_data["dxAtSpecimenAcquisition"].value_counts(dropna=False)

dxAtSpecimenAcquisition
ACUTE MYELOID LEUKAEMIA (AML) AND RELATED PRECURSOR NEOPLASMS    905
MYELODYSPLASTIC SYNDROMES                                         18
ACUTE LEUKAEMIAS OF AMBIGUOUS LINEAGE                              8
MYELODYSPLASTIC/MYELOPROLIFERATIVE NEOPLASMS                       7
MYELOPROLIFERATIVE NEOPLASMS                                       3
MATURE B-CELL NEOPLASMS                                            1
Name: count, dtype: int64

Da ich die Assoziation in primären AML-Zellen analysieren möchte, filtere ich anhand der Diagnose auf ACUTE MYELOID LEUKAEMIA (AML) AND RELATED PRECURSOR NEOPLASMS. Allerdings beeinhaltet diese Gruppe ggf. auch verwandte Neoplasien, dies werde ich in der deskriptiven Analyse weiter auswerten.

In [19]:
id_summary(clinical_data)

,Anzahl
n Patienten,805
n RNA Seq,699


In [20]:
# Filterung auf Diagnosen
clinical_data = clinical_data[clinical_data["dxAtSpecimenAcquisition"].eq("ACUTE MYELOID LEUKAEMIA (AML) AND RELATED PRECURSOR NEOPLASMS")]

In [21]:
id_summary(clinical_data)

,Anzahl
n Patienten,771
n RNA Seq,679


### RNA-Daten

Die Genexpressionsmatrix enthält Expressionswerte für zahlreiche Gene in den jeweiligen Proben. Für meine Analyse interessiert mich die Expression des Gens GFI1B, dafür extrahiere ich die Expressionswerte von GFI1B und ordne sie den entsprechenden RNA-IDs zu.

Anschließend transponiere ich die Matrix, damit die RNA-IDs als Zeilen vorliegen und sich die Expressionswerte einfacher mit den anderen Daten verknüpfen lassen.

In [22]:
rna_data = read_txt_file("beataml_waves1to4_norm_exp_dbgap.txt")

rna_data.shape

(22843, 711)

In [23]:
data_summary(rna_data)

,stable_id,display_label,description,biotype,BA2392R,BA2611R,BA2506R,BA2430R,BA2448R,BA2730R,BA2845R,BA2326R,BA2341R,BA2612R,BA2973R,BA2390R,BA3218R,BA3105R,BA3402R,BA3276R,BA2409R,BA3037R,BA2496R,BA2172R,BA2865R,BA2923R,BA2751R,BA2480R,BA2286R,BA2922R,BA2452R,BA3379R,BA2887R,BA2712R,BA2068R,BA2236R,BA2104R,BA3124R,BA3395R,BA2921R,BA2449R,BA2003R,BA2360R,BA2802R,BA2235R,BA2858R,BA3038R,BA3006R,BA2245R,BA2667R,BA2239R,BA2782R,BA2649R,BA2488R,BA2861R,BA2909R,BA2429R,BA2653R,BA2134R,BA2947R,BA2032R,BA2162R,BA2564R,BA2597R,BA3243R,BA2106R,BA2427R,BA2841R,BA2446R,BA2645R,BA2525R,BA2876R,BA2357R,BA2546R,BA2742R,BA2316R,BA2421R,BA2087R,BA2910R,BA2148R,BA2063R,BA2329R,BA3178R,BA2072R,BA3029R,BA2733R,BA2995R,BA2069R,BA2100R,BA2681R,BA2248R,BA2832R,BA2308R,BA2413R,BA2701R,BA2634R,BA2677R,BA2937R,BA2600R,BA2019R,BA2412R,BA2334R,BA2050R,BA3230R,BA2953R,BA2481R,BA2731R,BA2294R,BA2418R,BA2816R,BA2656R,BA2814R,BA2631R,BA2023R,BA2375R,BA2485R,BA2455R,BA2025R,BA2636R,BA2325R,BA2169R,BA2156R,BA2511R,BA2939R,BA2168R,BA2505R,BA2860R,BA2707R,BA2871R,BA2655R,BA2227R,BA2125R,BA2523R,BA2603R,BA2808R,BA2282R,BA2482R,BA2812R,BA2822R,BA2594R,BA2747R,BA2561R,BA2567R,BA2906R,BA2956R,BA2474R,BA2737R,BA2110R,BA2992R,BA2220R,BA2772R,BA2935R,BA2260R,BA2676R,BA2948R,BA2524R,BA2811R,BA2540R,BA2928R,BA2302R,BA2595R,BA2792R,BA2651R,BA2619R,BA3013R,BA2405R,BA2735R,BA2856R,BA2377R,BA2499R,BA2463R,BA3023R,BA2095R,BA2901R,BA2275R,BA2387R,BA2962R,BA2195R,BA2688R,BA2787R,BA2420R,BA2756R,BA2122R,BA2053R,BA2080R,BA2081R,BA2354R,BA3273R,BA2893R,BA2044R,BA2489R,BA2309R,BA2720R,BA2908R,BA2054R,BA2872R,BA2526R,BA2041R,BA2464R,BA2902R,BA2541R,BA2143R,BA2094R,BA2769R,BA2843R,BA2592R,BA2497R,BA2718R,BA2575R,BA2096R,BA2593R,BA2791R,BA2620R,BA2629R,BA2870R,BA2423R,BA2682R,BA2276R,BA2348R,BA2709R,BA2789R,BA2694R,BA2538R,BA2583R,BA2415R,BA2722R,BA2806R,BA2221R,BA3025R,BA2514R,BA2520R,BA2441R,BA2153R,BA2479R,BA2281R,BA2657R,BA2932R,BA2123R,BA2844R,BA2660R,BA2836R,BA3027R,BA2714R,BA2139R,BA2251R,BA2952R,BA2440R,BA2036R,BA2828R,BA2292R,BA2410R,BA2051R,BA2532R,BA2986R,BA2494R,BA2515R,BA2748R,BA2970R,BA2622R,BA2839R,BA2052R,BA3011R,BA2868R,BA2510R,BA2065R,BA2691R,BA2374R,BA2804R,BA2670R,BA2675R,BA2589R,BA2031R,BA2198R,BA2066R,BA2344R,BA2459R,BA2229R,BA2559R,BA2093R,BA2000R,BA2425R,BA2905R,BA2304R,BA2976R,BA2386R,BA2035R,BA2760R,BA2201R,BA2355R,BA2113R,BA2155R,BA2955R,BA2502R,BA2322R,BA2376R,BA2020R,BA2226R,BA2200R,BA2121R,BA2944R,BA2033R,BA2867R,BA2089R,BA2224R,BA2046R,BA2972R,BA2978R,BA2829R,BA2851R,BA2587R,BA2461R,BA2695R,BA2623R,BA2024R,BA2426R,BA2271R,BA3004R,BA2579R,BA3010R,BA2662R,BA2674R,BA2280R,BA2512R,BA2658R,BA2394R,BA2950R,BA2981R,BA2934R,BA2539R,BA2249R,BA2034R,BA2109R,BA2242R,BA2277R,BA2727R,BA2132R,BA2618R,BA2521R,BA2846R,BA2885R,BA2639R,BA2775R,BA2402R,BA2097R,BA2786R,BA2399R,BA2850R,BA2174R,BA2004R,BA3001R,BA2311R,BA2483R,BA3040R,BA2837R,BA2724R,BA2911R,BA2256R,BA2821R,BA2233R,BA2744R,BA2880R,BA2621R,BA3002R,BA2501R,BA2993R,BA2301R,BA2852R,BA2133R,BA2866R,BA2212R,BA2685R,BA2273R,BA2477R,BA2659R,BA2124R,BA2363R,BA2369R,BA2534R,BA2400R,BA2762R,BA2312R,BA2813R,BA2211R,BA2171R,BA2746R,BA2798R,BA2809R,BA2288R,BA2801R,BA3030R,BA2588R,BA2296R,BA2300R,BA2903R,BA2473R,BA2545R,BA2840R,BA2528R,BA2646R,BA3407R,BA2240R,BA2926R,BA2272R,BA2306R,BA2765R,BA2904R,BA2883R,BA2726R,BA2234R,BA2022R,BA2228R,BA2037R,BA2184R,BA3009R,BA2842R,BA2442R,BA2882R,BA2758R,BA2705R,BA2663R,BA2570R,BA2764R,BA2209R,BA2989R,BA2855R,BA2261R,BA2721R,BA2930R,BA2728R,BA2715R,BA2469R,BA2269R,BA2098R,BA2778R,BA2951R,BA2163R,BA2379R,BA2817R,BA3153R,BA2820R,BA2723R,BA2704R,BA2088R,BA2116R,BA2584R,BA3079R,BA2417R,BA2361R,BA2450R,BA2492R,BA2552R,BA2472R,BA2994R,BA2047R,BA2101R,BA2082R,BA2879R,BA2577R,BA2456R,BA2175R,BA2824R,BA2507R,BA2563R,BA2244R,BA2470R,BA2984R,BA2414R,BA2697R,BA2079R,BA2999R,BA2644R,BA2011R,BA2217R,BA2454R,BA2776R,BA2218R,BA2135R,BA2914R,BA2500R,BA2830R,BA2406R,BA3108R,BA2924R,BA2303R,BA2160R,BA2038R,BA2602R,BA2964R,BA2898R,BA3003R,BA2111R,BA2827R,BA2912R,BA2437R,BA2283R,BA2057R,BA2633R,BA3034R,BA2

In [24]:
# Filtern der RNA-Daten auf die Expression von GFI1B
rna_data_GFI1B = rna_data[rna_data.display_label == "GFI1B"]

# Transponieren und Formatierung der Daten um eine Tabelle mit den Spalten RNA-ID und GFI1B-Expression zu erhalten
rna_data_GFI1B = (rna_data_GFI1B
                  .drop(columns=["stable_id", "description", "biotype"])
                  .set_index("display_label")
                  .T
                  .reset_index()
                  .rename(columns={"index": "dbgap_rnaseq_sample"}))

del rna_data

rna_data_GFI1B.shape

(707, 2)

In [25]:
data_summary(rna_data_GFI1B)

display_label,dbgap_rnaseq_sample,GFI1B
dtype,str,float64
n missing,0,0
p% missing,0.0,0.0
n unique,707,686


### Score-Daten

Die Score-Daten enthalten für die jeweiligen RNA-IDs Scores für verschiedene maligne Zellzustände. Diese Scores beschreiben anhand von Genexpressionsmustern, wie stark bestimmte Zellzustände in den Proben ausgeprägt sind.
Für meine Analyse ist der Monocyte-like-Score relevant. Er beschreibt die Ausprägung eines monocytenähnlichen Genexpressionsprogramms. Da monocytische Differenzierung in AML mit einer geringeren Empfindlichkeit gegenüber Venetoclax in Verbindung gebracht wurde, untersuche ich den Score in einer zusätzlichen Sensitivitätsanalyse.

In [26]:
score_data = read_excel("beataml2_manuscript_vg_cts.xlsx")

# Umbenennen der Spalte da die Modelldefinitionen nicht mit - im Variablennamen umgehen können
score_data = score_data.rename(columns={"Monocyte-like": "monocyte_like"})

score_data.shape

(560, 7)

In [27]:
data_summary(score_data)

,dbgap_rnaseq_sample,GMP-like,HSC-like,monocyte_like,Progenitor-like,Promono-like,cDC-like
dtype,str,float64,float64,float64,float64,float64,float64
n missing,0,0,0,0,0,0,0
p% missing,0.0,0.0,0.0,0.0,0.0,0.0,0.0
n unique,560,560,560,560,560,560,560


### Tabellenverknüpfungen

Die verschiedenen Tabellen werden anhand der zugehörigen IDs zusammengeführt. Bei einem Inner-Merge bleiben nur Zeilen erhalten, für die es in beiden Tabellen eine passende ID gibt. Dadurch kann sich die Anzahl der Zeilen verringern.
Bei einem Left-Merge bleiben alle Zeilen der linken Tabelle erhalten. Die Informationen aus der rechten Tabelle werden anhand der IDs ergänzt. Die Zeilenanzahl bleibt dabei gleich, sofern die IDs in der rechten Tabelle eindeutig sind.

In [28]:
merge_summary = get_sample_summary(sample_df=sample_data)

In [29]:
data, merge_summary = merge_data(df_left=sample_data,
                                 df_right=response_data,
                                 merge_ids=["dbgap_subject_id", "dbgap_rnaseq_sample"],
                                 merge_summary=merge_summary,
                                 name="Response-Daten",
                                 inner=True)
del sample_data

data.shape

(433, 36)

In [30]:
data, merge_summary = merge_data(df_left=data,
                                 df_right=rna_data_GFI1B,
                                 merge_ids=["dbgap_rnaseq_sample"],
                                 merge_summary=merge_summary,
                                 name="RNA-Daten",
                                 inner=True)
del rna_data_GFI1B

data.shape

(367, 37)

In [31]:
data, merge_summary = merge_data(df_left=data,
                                 df_right=clinical_data,
                                 merge_ids=["dbgap_subject_id", "dbgap_rnaseq_sample"],
                                 merge_summary=merge_summary,
                                 name="Klinische Daten",
                                 inner=True)
del clinical_data

data.shape

(359, 130)

In [32]:
data, merge_summary = merge_data(df_left=data,
                                 df_right=score_data,
                                 merge_ids=["dbgap_rnaseq_sample"],
                                 merge_summary=merge_summary,
                                 name="Score-Daten",
                                 inner=False)
del score_data

data.shape

(359, 136)

In [33]:
merge_summary

,n Samples,n Patienten,n RNA Seq
Sample-IDs,942,805,698
Response-Daten,433,390,382
RNA-Daten,367,338,367
Klinische Daten,359,330,359
Score-Daten,359,330,359


In der Tabelle ist zu sehen, wie sich die Anzahl der Samples, Patienten und Expressionszeilen mit jedem Merge-Schritt verändert hat.

Durch das Mergen der Sample-IDs mit den Response-Daten blieben nur die Samples erhalten, für die Venetoclax-Daten vorliegen. Im nächsten Schritt wurde mit den RNA-Daten gemergt, dadurch fielen die Samples weg, für die keine RNA-Expressionsdaten vorliegen. Die klinischen Daten wurden bereits auf AML-Proben gefiltert, durch das Mergen mit diesen Daten verringerte sich die Anzahl daher erneut.

Lediglich durch das Hinzufügen der Score-Daten veränderten sich die Anzahlen nicht, da hierfür ein Left-Merge verwendet wurde. Dabei blieben alle Zeilen der linken Tabelle erhalten, auch wenn für einzelne RNA-IDs keine passenden Score-Daten vorlagen.

### Definition der Analysepopulation

Nach dem Zusammenführen der Tabellen prüfe ich, ob die benötigten Daten für die Analyse vorliegen. Dazu gehören die Venetoclax-AUC, die GFI1B-Expression sowie die klinischen Informationen. Außerdem prüfe ich, ob die verwendeten Dosis-Wirkungs-Modelle konvergiert sind und ob die Proben den festgelegten Auswahlkriterien entsprechen.

In [34]:
print("Duplikate Sample-IDs:", data["dbgap_rnaseq_sample"].duplicated().sum())
print("Duplikate Patienten-IDs:", data["dbgap_subject_id"].duplicated().sum())

Duplikate Sample-IDs: 0
Duplikate Patienten-IDs: 29


In [35]:
data["dbgap_subject_id"].value_counts().value_counts()

count
1    307
2     17
3      6
Name: count, dtype: int64

Um eine Abhängigkeit der Beobachtungen durch wiederholte Probenentnahmen derselben Patienten zu vermeiden, habe ich an dieser Stelle Patienten mit mehreren Proben aus der Analyse ausgeschlossen (n=23).

In [36]:
data = data.groupby("dbgap_subject_id").filter(lambda group: group["dbgap_rnaseq_sample"].nunique() == 1)

In [37]:
data["dbgap_subject_id"].value_counts().value_counts()

count
1    307
Name: count, dtype: int64

In [38]:
data["rna_control"].value_counts(dropna=False)

rna_control
No    307
Name: count, dtype: int64

Es liegen lediglich RNA Proben vor, die ausdrücklich keine gesunden Kontrollproben sind.

In [39]:
data["inhibitor"].value_counts(dropna=False)

inhibitor
Venetoclax    307
Name: count, dtype: int64

Alle Proben beziehen sich auf Venetoclax.

In [40]:
data["converged"].value_counts(dropna=False)

converged
True    307
Name: count, dtype: int64

Die Modelle für alle Proben sind konvergiert.

In [41]:
data_summary(data[["dbgap_subject_id", "dbgap_rnaseq_sample", "AUC", "GFI1B", "cohort",
                   "diseaseStageAtSpecimenCollection", "specificDxAtAcquisition", "monocyte_like"]])

,dbgap_subject_id,dbgap_rnaseq_sample,AUC,GFI1B,cohort,diseaseStageAtSpecimenCollection,specificDxAtAcquisition,monocyte_like
dtype,int64,str,float64,float64,str,str,str,float64
n missing,0,0,0,0,0,0,0,19
p% missing,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.19
n unique,307,307,304,303,3,5,18,288
